# 01 – Baseline: current PII detection on BPMN

This notebook runs the **existing, unchanged** code of
[lblod/pii-identification-anonymization-service](https://github.com/lblod/pii-identification-anonymization-service),
copied into `baseline_service/`, on a synthetic Dutch BPMN corpus (`data/synthetic`, made by `generate_bpmn.py`) and scores it against the known PII.
The numbers here are the baseline that later notebooks try to beat.

Setup: `bash setup.sh`, then `.venv/bin/python generate_bpmn.py`, and pick the kernel *Python (presidio-bpmn-tryout)*.

## 1. Setup: import the baseline service code

In [1]:
from importlib.metadata import version
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

# The service code (identification loads the spaCy models on import)
import baseline_service
from baseline_service.identification import analyzer
from baseline_service.endpoints import anonymize_bpmn as baseline_anonymize_bpmn
from baseline_service.endpoints import identify_bpmn as baseline_identify_bpmn

import pii_eval

print("baseline:", baseline_service.SOURCE_REPO, "@", baseline_service.SOURCE_COMMIT)
for pkg in ["presidio-analyzer", "spacy", "nl-core-news-md", "en-core-web-md"]:
    print(f"{pkg:20} {version(pkg)}")

baseline: https://github.com/lblod/pii-identification-anonymization-service @ d01e7ee
presidio-analyzer    2.2.358
spacy                3.8.16
nl-core-news-md      3.8.0
en-core-web-md       3.8.0


Which recognizers run? `detect_pii` always analyzes with `language="nl"` (the English fallback only triggers on a
`ValueError`). Presidio registers its language-agnostic pattern recognizers for `nl` too; the country-specific ones
(US SSN, UK NHS, credit cards, …) are English-only. The custom Belgian rijksregisternummer recognizer shows up as
`PatternRecognizer`.

In [2]:
def recognizer_table(lang):
    return pd.DataFrame(
        [{"recognizer": r.name, "entities": ", ".join(r.supported_entities)} for r in analyzer.get_recognizers(language=lang)]
    ).sort_values("recognizer").reset_index(drop=True)

recognizer_table("nl")

,recognizer,entities
0,CryptoRecognizer,CRYPTO
1,DateRecognizer,DATE_TIME
2,EmailRecognizer,EMAIL_ADDRESS
3,IbanRecognizer,IBAN_CODE
4,InVoterRecognizer,IN_VOTER
5,IpRecognizer,IP_ADDRESS
6,MedicalLicenseRecognizer,MEDICAL_LICENSE
7,PatternRecognizer,BELGIAN_NATIONAL_REGISTRY
8,PhoneRecognizer,PHONE_NUMBER
9,SpacyRecognizer,"ID, EMAIL, PHONE_NUMBER, NRP, LOCATION, DATE_TIME, AGE, PERSON"


`baseline_service.endpoints` holds the BPMN endpoint logic of the service's `web.py` without Flask:
`identify_bpmn(xml)` (= `POST /bpmn/identify`) and `anonymize_bpmn(xml, pointers)` (= `POST /bpmn/anonymize`).

## 2. The synthetic corpus

In [3]:
files, elements, spans = pii_eval.load_corpus("data/synthetic")
print(f"{len(files)} BPMN files, {len(elements)} text elements, {len(spans)} PII spans "
      f"in {(elements.n_pii > 0).sum()} elements")
pd.crosstab(spans.entity, spans.location, margins=True, margins_name="total")

20 BPMN files, 513 text elements, 187 PII spans in 142 elements


location,documentation,name,text,total
entity,,,,
ADDRESS,5,4,4,13
DATE_OF_BIRTH,3,4,0,7
EMAIL,6,7,4,17
IBAN,3,4,5,12
KBO,3,3,0,6
LICENSE_PLATE,3,4,0,7
PERSON,26,45,14,85
PHONE,9,8,5,22
RRN,5,8,5,18


Where does text sit in the files, and how much of it holds PII?

In [4]:
elements.assign(has_pii=elements.n_pii > 0).groupby("location").agg(
    elements=("text", "size"), with_pii=("has_pii", "sum"), pii_spans=("n_pii", "sum")
)

,elements,with_pii,pii_spans
location,,,
documentation,58,31,63
name,396,79,87
text,59,32,37


In [5]:
# A few examples per location
elements[elements.n_pii > 0].groupby("location").head(4)[["file", "location", "element_type", "text"]]

,file,location,element_type,text
1,aanvraag-rijbewijs-sintniklaas.bpmn,documentation,participant,Bij afwezigheid van Marleen Van de Velde neemt Hanne Jacobs het dossier over.
2,aanvraag-rijbewijs-sintniklaas.bpmn,name,participant,Jan Maes
4,aanvraag-rijbewijs-sintniklaas.bpmn,name,lane,Marleen - Burgerzaken
5,aanvraag-rijbewijs-sintniklaas.bpmn,name,lane,Marleen Van de Velde
7,aanvraag-rijbewijs-sintniklaas.bpmn,name,userTask,Advies vragen aan Jan Maes
11,aanvraag-rijbewijs-sintniklaas.bpmn,documentation,serviceTask,"Aanvrager Jan Maes, wonende te Schoolstraat 41A, 8800 Roeselare, geboren op 17/04/1958, rijksregisternummer 580417 4..."
16,aanvraag-rijbewijs-sintniklaas.bpmn,documentation,manualTask,"Testdossier: INSZ 670620 907 39, adres Leopoldlaan 18, 3000 Leuven."
23,aanvraag-rijbewijs-sintniklaas.bpmn,text,textAnnotation,Navragen bij Marleen
25,aanvraag-rijbewijs-sintniklaas.bpmn,text,textAnnotation,TODO: Jan Maes nog bevestigen
29,fuifkwaliteitslabel-aalter.bpmn,documentation,participant,Vragen? Mail llambrecht@aalter.be of bel +32 9 645 32 76 tijdens de kantooruren.


## 3. Run the baseline

In [6]:
%%time
baseline_outputs = {f: baseline_identify_bpmn(xml) for f, xml in files.items()}

detections = pd.DataFrame([
    {"file": f, "element_id": d["id"], "location": "name", **{k: v for k, v in d.items() if k != "id"}}
    for f, out in baseline_outputs.items() for d in out["pii_results"]
])
print(f"{len(detections)} detections, {sum(len(o['extracted_elements']) for o in baseline_outputs.values())} name attributes scanned")
detections.entity.value_counts()

136 detections, 396 name attributes scanned
CPU times: user 1.61 s, sys: 28.5 ms, total: 1.64 s
Wall time: 1.78 s


entity
PERSON                       49
LOCATION                     37
DATE_TIME                    30
EMAIL_ADDRESS                 7
BELGIAN_NATIONAL_REGISTRY     5
IBAN_CODE                     4
NRP                           4
Name: count, dtype: int64

## 4. Evaluation

In [7]:
scored_spans, scored_det = pii_eval.match(spans, detections)
pii_eval.summarize(scored_spans, scored_det)

,n_pii,n_detections,false_positives,precision,recall,recall_full,recall_typed,f1
scope,,,,,,,,
all,187,136,51,0.625,0.433,0.342,0.358,0.512


- **recall**: share of PII spans touched by a detection; **recall_full**: share completely covered (what
  masking needs, e.g. `Lander Lambrecht` with only `Lambrecht` detected is touched but not covered);
  **recall_typed**: touched by a detection of the right type.
- **precision**: share of detections that touch real PII.

### Per location

In [8]:
pii_eval.summarize(scored_spans, scored_det, by="location")

,n_pii,n_detections,false_positives,precision,recall,recall_full,recall_typed,f1
location,,,,,,,,
documentation,63,0,0,NaN,0.000,0.000,0.00,NaN
name,87,136,51,0.625,0.931,0.736,0.77,0.748
text,37,0,0,NaN,0.000,0.000,0.00,NaN


The service only reads `name` attributes, so PII in `<bpmn:documentation>` and text annotations (`<bpmn:text>`)
is never found.

### Per entity type

In [9]:
pii_eval.summarize(scored_spans, scored_det, by="entity")

,n_pii,recall,recall_full,recall_typed
entity,,,,
ADDRESS,13,0.308,0.000,0.308
DATE_OF_BIRTH,7,0.571,0.571,0.571
EMAIL,17,0.412,0.412,0.412
IBAN,12,0.333,0.333,0.333
KBO,6,0.000,0.000,0.000
LICENSE_PLATE,7,0.286,0.286,0.000
PERSON,85,0.529,0.494,0.529
PHONE,22,0.318,0.091,0.000
RRN,18,0.444,0.167,0.167


Same, restricted to `name` attributes (the part the service does scan):

In [10]:
in_names = scored_spans.location == "name"
pii_eval.summarize(scored_spans[in_names], scored_det, by="entity")

,n_pii,recall,recall_full,recall_typed
entity,,,,
ADDRESS,4,1.000,0.000,1.000
DATE_OF_BIRTH,4,1.000,1.000,1.000
EMAIL,7,1.000,1.000,1.000
IBAN,4,1.000,1.000,1.000
KBO,3,0.000,0.000,0.000
LICENSE_PLATE,4,0.500,0.500,0.000
PERSON,45,1.000,0.933,1.000
PHONE,8,0.875,0.250,0.000
RRN,8,1.000,0.375,0.375


### Missed or partially covered PII in `name` attributes

In [11]:
scored_spans[in_names & ~scored_spans.full][["entity", "value", "hit", "text"]].sort_values("entity")

,entity,value,hit,text
161,ADDRESS,"Schoolstraat 97 bus 2, 2500 Lier",True,"Plaatsbezoek Schoolstraat 97 bus 2, 2500 Lier"
23,ADDRESS,"Veldstraat 11, 8800 Roeselare",True,"Plaatsbezoek Veldstraat 11, 8800 Roeselare"
62,ADDRESS,"Stationsstraat 35, 8000 Brugge",True,"Plaatsbezoek Stationsstraat 35, 8000 Brugge"
128,ADDRESS,"Kerkstraat 167, 2300 Turnhout",True,"Plaatsbezoek Kerkstraat 167, 2300 Turnhout"
54,KBO,0700.775.411,False,Facturatie KBO 0700.775.411
108,KBO,0725.555.347,False,Facturatie KBO 0725.555.347
35,KBO,0585.669.865,False,Facturatie KBO 0585.669.865
87,LICENSE_PLATE,2-QUA-787,False,Nummerplaat 2-QUA-787 registreren
166,LICENSE_PLATE,2-UAT-004,False,Nummerplaat 2-UAT-004 registreren
24,PERSON,Lander Lambrecht,True,Doorsturen naar Lander Lambrecht


For context, what did the baseline return on those elements?

In [12]:
partial = scored_spans[in_names & scored_spans.hit & ~scored_spans.full]
partial[pii_eval.KEY].drop_duplicates().merge(scored_det, on=pii_eval.KEY)[["entity", "match", "score", "gt_entity"]]

,entity,match,score,gt_entity
0,DATE_TIME,670620,0.85,RRN
1,PERSON,Lambrecht,0.85,PERSON
2,DATE_TIME,786,0.85,RRN
3,LOCATION,Roeselare,0.85,ADDRESS
4,PERSON,Lambrecht,0.85,PERSON
5,PERSON,Koen,0.85,PERSON
6,BELGIAN_NATIONAL_REGISTRY,32495454625,0.85,PHONE
7,PERSON,Plaatsbezoek,0.85,None
8,LOCATION,Brugge,0.85,ADDRESS
9,DATE_TIME,051,0.85,PHONE


### False positives

In [13]:
fp = scored_det[~scored_det.tp].merge(elements[pii_eval.KEY + ["text"]], on=pii_eval.KEY)
print(f"{len(fp)} false positives")
fp[["entity", "match", "score", "text"]].sort_values("entity")

51 false positives


,entity,match,score,text
26,DATE_TIME,1 maand,0.85,1 maand voor de fuif
27,DATE_TIME,31 augustus,0.85,Uiterlijk 31 augustus
28,DATE_TIME,Aangifte,0.85,Aangifte huwelijk Genk
3,DATE_TIME,30 dagen,0.85,Binnen 30 dagen
42,DATE_TIME,20250226719,0.85,Dossiernummer 20250226719 aanmaken
19,DATE_TIME,30 dagen,0.85,Binnen 30 dagen
18,DATE_TIME,3 maanden,0.85,Aanvraag evenement minstens 3 maanden vooraf
30,DATE_TIME,Aangifte,0.85,Aangifte huwelijk Hasselt
15,DATE_TIME,3 maanden,0.85,Aanvraag evenement minstens 3 maanden vooraf
23,DATE_TIME,1 maand,0.85,1 maand voor de fuif


### Detections with the wrong type

In [14]:
scored_det[scored_det.tp & ~scored_det.tp_typed][["entity", "gt_entity", "match", "score"]]

,entity,gt_entity,match,score
2,DATE_TIME,RRN,670620,0.85
27,DATE_TIME,LICENSE_PLATE,2-NDY-243,0.85
59,DATE_TIME,RRN,786,0.85
65,DATE_TIME,RRN,000501,0.85
66,DATE_TIME,RRN,209,0.85
67,DATE_TIME,LICENSE_PLATE,1-MSV-261,0.85
74,DATE_TIME,PHONE,460,0.85
75,DATE_TIME,PHONE,46 54 54,0.85
81,DATE_TIME,PHONE,460,0.85
82,DATE_TIME,PHONE,46 54 54,0.85


## 5. Anonymization round trip

Feed every baseline detection back into the `/bpmn/anonymize` logic, then check per PII span what is left of it
in the anonymized element text: **leaked** = at least one word or number of the value survives
(e.g. `Lander *****`), **leaked_intact** = the whole value survives.

In [15]:
anonymized = {f: baseline_anonymize_bpmn(xml, baseline_outputs[f]["pii_results"]) for f, xml in files.items()}
residual = pii_eval.residual_pii(spans, anonymized)
print(f"PII (partly) left after anonymization: {residual.leaked.sum()} / {len(residual)} ({residual.leaked.mean():.0%}), "
      f"fully intact: {residual.leaked_intact.sum()}")
residual.groupby("location")[["leaked", "leaked_intact"]].agg(["sum", "mean"]).round(3)

PII (partly) left after anonymization: 120 / 187 (64%), fully intact: 106


leaked       leaked_intact       
                 sum  mean           sum   mean
location                                       
documentation     63  1.00            63  1.000
name              20  0.23             6  0.069
text              37  1.00            37  1.000

In [16]:
example = "fuifkwaliteitslabel-aalter.bpmn"
before = elements[elements.file == example][pii_eval.KEY + ["text"]]
root = pii_eval.ET.fromstring(anonymized[example])
before.assign(after=[pii_eval.element_text(root, e, loc) for e, loc in zip(before.element_id, before.location)]) \
      .loc[lambda d: d.text != d.after, ["location", "text", "after"]]

,location,text,after
28,name,Jeugd - Fuifkwaliteitslabel aanvraag Aalter,Jeugd - ***** aanvraag *****
32,name,Lander Lambrecht,Lander *****
35,name,1 maand voor de fuif,***** voor de fuif
36,name,Goedkeuring door dhr. Lambrecht,Goedkeuring door *****
38,name,Rijksregister 050407 786 39 opvragen,Rijksregister 050407 ***** 39 opvragen
39,name,"Plaatsbezoek Veldstraat 11, 8800 Roeselare","Plaatsbezoek Veldstraat 11, 8800 *****"
40,name,Doorsturen naar Lander Lambrecht,Doorsturen naar Lander *****


PII in that file that is (partly) left in the output:

In [17]:
residual[(residual.file == example) & residual.leaked][["location", "entity", "value", "leaked_intact", "text"]]

,location,entity,value,leaked_intact,text
18,documentation,EMAIL,llambrecht@aalter.be,True,Vragen? Mail llambrecht@aalter.be of bel +32 9 645 32 76 tijdens de kantooruren.
19,documentation,PHONE,+32 9 645 32 76,True,Vragen? Mail llambrecht@aalter.be of bel +32 9 645 32 76 tijdens de kantooruren.
20,name,PERSON,Lander Lambrecht,False,Lander Lambrecht
22,name,RRN,050407 786 39,False,Rijksregister 050407 786 39 opvragen
23,name,ADDRESS,"Veldstraat 11, 8800 Roeselare",False,"Plaatsbezoek Veldstraat 11, 8800 Roeselare"
24,name,PERSON,Lander Lambrecht,False,Doorsturen naar Lander Lambrecht
25,text,RRN,970804-277-47,True,INSZ 970804-277-47


The serialized XML loses the original namespace prefixes (`bpmn:` becomes `ns0:`):

In [18]:
print(files[example][:300])
print("...")
print(anonymized[example][:300])

<?xml version='1.0' encoding='UTF-8'?>
<bpmn:definitions xmlns:bpmn="http://www.omg.org/spec/BPMN/20100524/MODEL" id="Definitions_3yvb5ul" targetNamespace="http://bpmn.io/schema/bpmn" exporter="Camunda Modeler" exporterVersion="5.14.0">
  <bpmn:collaboration id="Collaboration_9a7mfp0">
    <bpmn:par
...
<?xml version='1.0' encoding='utf-8'?>
<ns0:definitions xmlns:ns0="http://www.omg.org/spec/BPMN/20100524/MODEL" id="Definitions_3yvb5ul" targetNamespace="http://bpmn.io/schema/bpmn" exporter="Camunda Modeler" exporterVersion="5.14.0">
  <ns0:collaboration id="Collaboration_9a7mfp0">
    <ns0:partici


## 6. Baseline summary

In [19]:
summary = pd.concat([
    pii_eval.summarize(scored_spans, scored_det).assign(group="overall"),
    pii_eval.summarize(scored_spans, scored_det, by="location").assign(group="location"),
    pii_eval.summarize(scored_spans, scored_det, by="entity").assign(group="entity"),
]).rename_axis("scope").reset_index()
summary.insert(0, "approach", "baseline")
summary["residual_pii"] = summary["scope"].map(
    {"all": residual.leaked.mean(), **residual.groupby("location").leaked.mean(), **residual.groupby("entity").leaked.mean()}
).round(3)

Path("results").mkdir(exist_ok=True)
summary.to_csv("results/baseline.csv", index=False)
summary

,approach,scope,n_pii,n_detections,false_positives,precision,recall,recall_full,recall_typed,f1,group,residual_pii
0,baseline,all,187,136.0,51.0,0.625,0.433,0.342,0.358,0.512,overall,0.642
1,baseline,documentation,63,0.0,0.0,NaN,0.000,0.000,0.000,NaN,location,1.000
2,baseline,name,87,136.0,51.0,0.625,0.931,0.736,0.770,0.748,location,0.230
3,baseline,text,37,0.0,0.0,NaN,0.000,0.000,0.000,NaN,location,1.000
4,baseline,ADDRESS,13,NaN,NaN,NaN,0.308,0.000,0.308,NaN,entity,1.000
5,baseline,DATE_OF_BIRTH,7,NaN,NaN,NaN,0.571,0.571,0.571,NaN,entity,0.429
6,baseline,EMAIL,17,NaN,NaN,NaN,0.412,0.412,0.412,NaN,entity,0.588
7,baseline,IBAN,12,NaN,NaN,NaN,0.333,0.333,0.333,NaN,entity,0.667
8,baseline,KBO,6,NaN,NaN,NaN,0.000,0.000,0.000,NaN,entity,1.000
9,baseline,LICENSE_PLATE,7,NaN,NaN,NaN,0.286,0.286,0.000,NaN,entity,0.714


### Where to improve next

The tables above point at the main gaps; candidate steps for the next notebooks:

1. **Coverage**: also scan `<bpmn:documentation>` and `<bpmn:textAnnotation>/<bpmn:text>` (and keep the anonymizer
   writing back to those locations, with the original namespace prefixes).
2. **Belgian formats**: phone numbers in Belgian notation, rijksregisternummer written with spaces, KBO numbers,
   license plates, addresses (street + number + postcode + municipality).
3. **Precision**: the 11-digit RRN regex fires on case numbers; validate the mod-97 checksum and use context words.
4. **Names**: partial name spans (first name missed) and short first-name-only mentions.